In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, isnull, lit
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re
from pyspark.sql.window import Window


StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 3, Finished, Available, Finished, False)

In [2]:
# ======================================================================================
# Helper Functions
# ======================================================================================

def clean_spark_cols(df):
    """Replaces special characters in column names with underscores."""
    new_columns = [re.sub(r'[\s\.\-\(\)]', '_', c) for c in df.columns]
    return df.toDF(*new_columns)

def cleanse_dataframe(df):
    """
    Applies a series of cleaning transformations to a Spark DataFrame,
    replicating the logic of the Alteryx Cleanse tool.
    
    - Replaces nulls with 0 for numeric fields.
    - Replaces nulls with blanks for string fields.
    - Trims leading/trailing whitespace.
    - Removes tabs, line breaks, and duplicate whitespace.
    - Converts all string fields to uppercase.
    """
    print("Applying cleansing transformations...")
    
    # Get lists of column names based on their data type
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType, DecimalType))]

    # Start with the dataframe to be cleansed
    cleansed_df = df

    # Replace nulls with 0 for all numeric fields
    cleansed_df = cleansed_df.na.fill(0, subset=numeric_cols)

    # Apply a series of cleaning transformations to all string fields
    for col_name in string_cols:
        cleansed_df = cleansed_df.withColumn(
            col_name,
            F.upper(                                                     # 4. Make all of them upper case
                F.regexp_replace(                                        # 3c. Remove duplicate whitespace
                    F.regexp_replace(                                    # 3b. Remove tabs and line breaks
                        F.trim(                                          # 3a. Remove leading and trailing whitespace
                            F.coalesce(F.col(col_name), F.lit(''))        # 2. Replace nulls with blanks
                        ),
                        r'[\t\n\r]', ''
                    ),
                    r'\s+', ' '
                )
            )
        )

    # Specific cleaning for 'Likelihood_of_Win' if it exists, as it's a special case
    if 'Likelihood_of_Win' in cleansed_df.columns:
        cleansed_df = cleansed_df.withColumn("Likelihood_of_Win", F.regexp_replace(F.col("Likelihood_of_Win"), "%", "").cast(DoubleType()))

    print("Cleansing finished.")
    return cleansed_df

def advanced_preprocess(df, col_name):
    """Applies advanced text cleaning to a column for fuzzy matching."""
    # Transformations are nested to be applied sequentially.
    processed_col = F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.upper(F.col(col_name)),
                r'\b(INC|CORP|CORPORATION|LTD|LIMITED|LLC|CO|COMPANY|PLC)\b', ''
            ),
            r'[^\w\s]', ''
        )
    )
    return df.withColumn(f"{col_name}_processed", processed_col)

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 4, Finished, Available, Finished, False)

In [3]:

# ======================================================================================
# Batch 1: Data Ingestion and Initial Joins
# --------------------------------------------------------------------------------------
# In a real Fabric environment, you would replace these file paths with reads from your
# Lakehouse tables or files in OneLake.
# Example: spark.read.format("delta").load("Tables/YourTableName")
# ======================================================================================
print("Batch 1: Ingesting data and performing initial joins...")

# Import all files".
legacy_cis_crb_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Legacy CIS`")
sales_forecast = spark.sql("""
SELECT 
    `DunsNumber`,
    `OriginatingLeadID`,
    `GUID`,
    `OpptyID`,
    `OpptySummary`,
    `Pipeline Phase`,
    `Description`,
    `FormName`,
    `ProductClass`,
    `ProductSubClass`,
    `Frequency`,
    `OpptyStatus`,
    `OpptyState`,
    `OpptyType`,
    `OpptySubType`,
    `Est. Revenue (LCY)`,
    `Est. Revenue (USD)`,
    `Wt. Revenue (LCY)`,
    `Wt. Revenue (USD)`,
    `CCY`,
    `Likelihood of Win`,
    `Premium (USD)`,
    `Project Start Date`,
    `ModifiedOn`,
    `CreatedOn`,
    `CloseDate`,
    `Source Campaign`,
    `Account`,
    `Parent Account`,
    `Global Parent Account`,
    `GCID`,
    `Tiers`,
    `Service Region`,
    `Service Office`,
    `Service Office Country`,
    `Profit Center`,
    `Finance Level`,
    `Opportunity Owner`,
    `Opportunity Owner UPN`,
    `Colleague Involved`,
    `Colleague Involved Office`,
    `Colleague Involved UPN`,
    `OwnerProfitCenter`,
    `TagList`,
    `First Income Date`,
    `Industry (Account Name) (Account)`,
    `Primary SIC Code (Account Name)(Account)`,
    `Country (Account Name)(Account)`,
    `Data Version`
FROM APAC_CRM_Analytics_LH.`All lobs_OS`

UNION

SELECT 
    `DunsNumber`,
    `OriginatingLeadID`,
    `GUID`,
    `OpptyID`,
    `OpptySummary`,
    `Pipeline Phase`,
    `Description`,
    `FormName`,
    `ProductClass`,
    `ProductSubClass`,
    `Frequency`,
    `OpptyStatus`,
    `OpptyState`,
    `OpptyType`,
    `OpptySubType`,
    `Est. Revenue (LCY)`,
    `Est. Revenue (USD)`,
    `Wt. Revenue (LCY)`,
    `Wt. Revenue (USD)`,
    `CCY`,
    `Likelihood of Win`,
    `Premium (USD)`,
    `Project Start Date`,
    `ModifiedOn`,
    `CreatedOn`,
    `CloseDate`,
    `Source Campaign`,
    `Account`,
    `Parent Account`,
    `Global Parent Account`,
    `GCID`,
    `Tiers`,
    `Service Region`,
    `Service Office`,
    `Service Office Country`,
    `Profit Center`,
    `Finance Level`,
    `Opportunity Owner`,
    `Opportunity Owner UPN`,
    `Colleague Involved`,
    `Colleague Involved Office`,
    `Colleague Involved UPN`,
    `OwnerProfitCenter`,
    `TagList`,
    `First Income Date`,
    `Industry (Account Name) (Account)`,
    `Primary SIC Code (Account Name)(Account)`,
    `Country (Account Name)(Account)`,
    `Data Version`
FROM APAC_CRM_Analytics_LH.`All Lobs_O`
""")
workers_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`rs_workers`")
profit_center_mapping_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Profit Center to Business Mapping`")
office_country_mapping_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Service Owner Office Mapping`")
product_mapping_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Product to Business Mapping`")
frequency_mapping_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Frequency Mapping`")
pipeline_phase_mapping_df = spark.sql("SELECT * FROM APAC_CRM_Analytics_LH.`Pipeline Phasing Mapping`")

# Apply column cleaning
sales_forecast_df = clean_spark_cols(sales_forecast)
legacy_cis_crb_df = clean_spark_cols(legacy_cis_crb_df)
workers_df = cleanse_dataframe(clean_spark_cols(workers_df))
profit_center_mapping_df = cleanse_dataframe(clean_spark_cols(profit_center_mapping_df))
office_country_mapping_df = cleanse_dataframe(clean_spark_cols(office_country_mapping_df))
product_mapping_df = cleanse_dataframe(clean_spark_cols(product_mapping_df))
frequency_mapping_df = cleanse_dataframe(clean_spark_cols(frequency_mapping_df))
pipeline_phase_mapping_df = cleanse_dataframe(clean_spark_cols(pipeline_phase_mapping_df))
# ---- END FIX ----

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 5, Finished, Available, Finished, False)

Batch 1: Ingesting data and performing initial joins...
Applying cleansing transformations...
Cleansing finished.
Applying cleansing transformations...
Cleansing finished.
Applying cleansing transformations...
Cleansing finished.
Applying cleansing transformations...
Cleansing finished.
Applying cleansing transformations...
Cleansing finished.
Applying cleansing transformations...
Cleansing finished.


In [4]:
sales_forecast_df = sales_forecast_df.withColumnRenamed("OwnerProfitCenter", "Opportunity_Owner_ProfitCenter")
display(sales_forecast_df)

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a851408e-5d51-4519-8814-d1865c65b220)

In [5]:
# This section implements the precise join and union logic from the Alteryx workflow.

# 1. Find records only in new sales data (left anti-join)
sales_only = sales_forecast_df.join(legacy_cis_crb_df, "OpptyID", "left_anti")

# 2. Find records only in legacy (right anti-join) 
legacy_only = legacy_cis_crb_df.join(sales_forecast_df, "OpptyID", "left_anti")

# 3. Find matching records (equivalent to the Join anchor) and construct them as specified.
# We prioritize legacy fields but take specific fields from the new sales data.
matched_join = legacy_cis_crb_df.alias("legacy").join(
    sales_forecast_df.alias("sales"),
    "OpptyID",
    "inner"
)

# Create a list of columns, taking all from legacy and adding the two specified columns from sales.
# The alias is important to avoid ambiguity.
select_exprs = [F.col(f"legacy.{c}") for c in legacy_cis_crb_df.columns] \
             + [F.col("sales.Opportunity_Owner_ProfitCenter"), F.col("sales.Opportunity_Owner_UPN")]

matched_records = matched_join.select(*select_exprs)


# 4. Schema alignment for the three DataFrames before unioning.
# This ensures all three parts have the exact same columns in the same order.
all_columns = set(sales_only.columns) | set(legacy_only.columns) | set(matched_records.columns)
final_columns_order = sorted(list(all_columns))

def align_and_select(df, column_order):
    """Adds missing columns as null and selects all columns in a specific order."""
    missing_cols = set(column_order) - set(df.columns)
    for col_name in missing_cols:
        df = df.withColumn(col_name, F.lit(None))
    return df.select(column_order)

sales_only_aligned = align_and_select(sales_only, final_columns_order)
legacy_only_aligned = align_and_select(legacy_only, final_columns_order)
matched_records_aligned = align_and_select(matched_records, final_columns_order)

# 5. Union all three aligned datasets (equivalent to the Alteryx Union tool)
combined_df = sales_only_aligned.union(legacy_only_aligned).union(matched_records_aligned)

# Filter for APAC Service Regions (equivalent to Alteryx Filter tool 266)
apac_regions = ["Asia", "Australasia", "Vietnam", "India", "China"]
combined_df = combined_df.filter(F.col("Service_Region").isin(apac_regions))

print("Batch 1 finished.")
print(f"sales_only count: {sales_only.count()}")
print(f"legacy_only count: {legacy_only.count()}")
print(f"matched_records count: {matched_records.count()}")
print(f"combined_df count: {combined_df.count()}")

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 7, Finished, Available, Finished, False)

Batch 1 finished.
sales_only count: 167944
legacy_only count: 48574
matched_records count: 90475
combined_df count: 287884


In [6]:
# ======================================================================================
# Batch 2: Data Cleansing and Standardization
# --------------------------------------------------------------------------------------
# This batch now calls the reusable cleanse_dataframe function.
# ======================================================================================
print("Batch 2: Starting data cleansing...")
cleansed_df = cleanse_dataframe(combined_df)
print("Batch 2 finished.")

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 8, Finished, Available, Finished, False)

Batch 2: Starting data cleansing...
Applying cleansing transformations...
Cleansing finished.
Batch 2 finished.


In [7]:
# ======================================================================================
# Batch 3: Data Enrichment and Mapping
# --------------------------------------------------------------------------------------
# Replicates the series of Joins and Formulas for mapping and enrichment.
# ======================================================================================
print("Batch 3: Enriching data through mapping...")

# Filter and get unique workers (equivalent to Alteryx Filter 249 and Unique 251)
unique_workers_df = workers_df.filter(
    (F.col("statuscodename") == "ACTIVE") & (F.col("Owner_Business_name2").isNotNull())
).dropDuplicates(["rs_upn"])

# Join with unique workers data and select/rename columns (equivalent to Alteryx Join tool 258)
enriched_df = cleansed_df.alias("left").join(
    unique_workers_df.alias("right"),
    # Add F.trim() to both sides of the comparison
    F.trim(F.upper(F.col("left.Colleague_Involved_UPN"))) == F.trim(F.upper(F.col("right.rs_upn"))),
    "left"  # This is a left join
).select(
    "left.*",
    F.col("right.Owner_Business_name2").alias("Owner_Business_Name"),
    F.col("right.Owner_Country_name2").alias("Owner_Country"),
    F.col("right.Owner_Lob_Name2").alias("Owner_Lob_Name"),
    F.col("right.Owner_Segment_Lob_Name2").alias("Owner_Segment_Lob_Name")
)

# Join with Profit Center mapping (equivalent to Alteryx Join tool 183)
enriched_df = enriched_df.alias("left").join(
    profit_center_mapping_df.alias("right"),
    # Apply upper and trim to both sides of the join condition
    F.upper(F.trim(F.col("left.Profit_Center"))) == F.upper(F.trim(F.col("right.Profit_Center"))),
    "left"
).select("left.*", "right.Map_to_Business_Name", "right.Map_to_Segment_Name")

# Formula Tool 215: Apply mappings and clean up columns
enriched_df = enriched_df.withColumn(
    # Use coalesce to fill Owner_Business_Name ONLY if it's null.
    "Owner_Business_Name",
    F.coalesce("Owner_Business_Name", "Map_to_Business_Name")
).withColumn(
    # Use coalesce to fill Owner_Segment_Lob_Name ONLY if it's null.
    "Owner_Segment_Lob_Name",
    F.coalesce("Owner_Segment_Lob_Name", "Map_to_Segment_Name")
).withColumn(
    "Finance_Level",
    # Chain all logic for the 'Finance_Level' column in one pass
    F.when(F.upper(F.trim(F.col("Finance_Level"))) == "VIET NAM", F.lit("VIETNAM"))
     .when(F.upper(F.trim(F.col("Finance_Level"))) == "KOREA (REPUBLIC OF)", F.lit("SOUTH KOREA"))
     .when(F.col("Finance_Level").isNull() | (F.trim(F.col("Finance_Level")) == ""), F.col("Service_Office_Country"))
     .otherwise(F.col("Finance_Level"))
)

# Join with Office Country mapping (equivalent to Alteryx Join tool 213)
enriched_df = enriched_df.alias("left").join(
    office_country_mapping_df.alias("right"),
    # Apply upper and trim to both sides of the join condition
    F.upper(F.trim(F.col("left.Finance_Level"))) == F.upper(F.trim(F.col("right.Finance_Level"))),
    "left"
).select("left.*", "right.Mapping_to_County")

# Formula Tool 184: Fill nulls and standardize country names
enriched_df = enriched_df.withColumn("Service_Office_Country",
                                     F.when(F.col("Service_Office_Country").isNull() | (F.col("Service_Office_Country") == ""), F.col("Mapping_to_County"))
                                     .when(F.col("Service_Office_Country").isin("KOREA, REPUBLIC OF", "KOREA (REPUBLIC OF)"), "SOUTH KOREA")
                                     .otherwise(F.col("Service_Office_Country")))

enriched_df = enriched_df.withColumn("Country__Account_Name__Account_",
                                   F.when(F.col("Country__Account_Name__Account_").isNull() | (F.col("Country__Account_Name__Account_") == ""), F.col("Service_Office_Country"))
                                   .otherwise(F.col("Country__Account_Name__Account_")))

# ---- Conditional Remapping Logic (Replicates Alteryx Filter 216, Join 229, Formula 230, Union 223) ----
# This is to map business for the rest of null from previous mapping
business_lines = ["CORPORATE RISK & BROKING", "HEALTH AND BENEFITS", "RETIREMENT", "WORK & REWARDS",
                  "EMPLOYEE EXPERIENCE", "INSURANCE CONSULTING AND TECHNOLOGY", "INVESTMENT BUSINESS",
                  "INTEGRATED AND GLOBAL SOLUTIONS"]

# Records that already have a valid business name
business_records_df = enriched_df.filter(
    F.col("Owner_Business_Name").isin(business_lines) &
    F.col("Owner_Business_Name").isNotNull()
)

# Records that need remapping
non_business_records_df = enriched_df.filter(
    (~F.col("Owner_Business_Name").isin(business_lines)) |
    F.col("Owner_Business_Name").isNull()
)

# Join the non-business records with the product mapping table
remapped_df = non_business_records_df.alias("left").join(
    product_mapping_df.alias("right"),
    F.col("left.ProductClass") == F.col("right.ProductClass"),
    "left"
).select(
    *[F.col(f"left.{c}") for c in non_business_records_df.columns if c not in ["Owner_Business_Name", "Owner_Segment_Lob_Name"]],
    F.col("right.Map_to_Business").alias("Owner_Business_Name"),
    F.col("right.Map_to_Segment").alias("Owner_Segment_Lob_Name")
)

# Union the two streams back together (Tool 223)
combined_after_remap_df = business_records_df.unionByName(remapped_df.select(business_records_df.columns))

# --- FINAL FORMULAS (Tool 270) ---
# Apply final logic to the fully combined and remapped dataframe
final_enriched_df = combined_after_remap_df.withColumn("Owner_Country", F.when(F.col("Owner_Country").isNull() | (F.col("Owner_Country") == ""), F.col("Service_Office_Country")).otherwise(F.col("Owner_Country"))) \
    .withColumn("Industry", F.when(F.col("Industry__Account_Name___Account_").isNull() | (F.col("Industry__Account_Name___Account_") == ""), "UNMAPPED").otherwise(F.col("Industry__Account_Name___Account_"))) \
    .withColumn("Frequency", F.when(F.col("Frequency").isNull() | (F.col("Frequency") == ""), "UNMAPPED").otherwise(F.col("Frequency"))) \
    .withColumn("OpptyType", F.when(F.col("OpptyType").isNull() | (F.col("OpptyType") == ""), "UNMAPPED").otherwise(F.col("OpptyType"))) \
    .withColumn("Pipeline_Phase",
        F.when(F.col("Pipeline_Phase").isNull() & F.col("OpptyState").contains("OPEN"), "1 - DISCOVER")
         .when(F.col("Pipeline_Phase").isNull() & F.col("OpptyState").contains("WON"), "5 - FINALISE")
         .otherwise(F.col("Pipeline_Phase")))

# Define the columns before joining to avoid reference issues
left_columns_except_frequency = [c for c in final_enriched_df.columns if c != "Frequency"]

# Join with Frequency mapping (equivalent to Alteryx Join tool 283)
final_enriched_df = final_enriched_df.alias("left").join(
    frequency_mapping_df.alias("right"),
    F.col("left.Frequency") == F.col("right.CRM_Frequency"),
    "left"
).select(
    *[F.col(f"left.{c}") for c in left_columns_except_frequency],
    F.col("right.Frequency")
)

# Define columns before pipeline phase join
left_columns_except_pipeline_phase = [c for c in final_enriched_df.columns if c != "Pipeline_Phase"]

# Join with Pipeline Phase mapping (equivalent to Alteryx Join tool 292)
final_enriched_df = final_enriched_df.alias("left").join(
    pipeline_phase_mapping_df.alias("right"),
    F.col("left.Pipeline_Phase") == F.col("right.D365_Pipeline_Phase"),
    "left"
).select(
    *[F.col(f"left.{c}") for c in left_columns_except_pipeline_phase],
    F.col("right.New_Pipeline_Phase").alias("Pipeline_Phase")
)

# --- Combined Formula Logic for tools 287, 342, 351 ---
final_enriched_df = final_enriched_df \
    .withColumn("n_GCID", F.when(F.col("GCID").isNull() | (F.col("GCID") == ""), F.col("Account")).otherwise(F.col("GCID"))) \
    .withColumn("Owner_Business_Name", F.when(F.col("Owner_Business_Name").isNull() | (F.col("Owner_Business_Name") == ""), "UNMAPPED").otherwise(F.col("Owner_Business_Name"))) \
    .withColumn("Owner_Segment_Lob_Name", F.when(F.col("Owner_Segment_Lob_Name").isNull() | (F.col("Owner_Segment_Lob_Name") == ""), "NON-OPERATING SEGMENT").otherwise(F.col("Owner_Segment_Lob_Name"))) \
    .withColumn("Opp_Link", F.concat(F.lit("https://wtwcrb.crm.dynamics.com/main.aspx?appid=af96b65c-0084-ea11-a813-000d3a579b83&forceUCI=1&pagetype=entityrecord&etn=opportunity&id="), F.col("GUID"))) \
    .withColumn("OpptySubType_Original", F.col("OpptySubType")) \
    .withColumn("OpptyType_Original", F.col("OpptyType")) \
    .withColumn("OpptySubType", F.when(F.col("OpptySubType") == "RENEWAL", "RENEWAL").otherwise("NEW BUSINESS")) \
    .withColumn("OpptyType",
         F.when(F.col("OpptyType_Original") == "GLOBAL FAC", F.col("OpptySubType_Original"))
         .when(F.col("OpptyType_Original") == "RENEWAL", "RENEWAL")
         .when(F.col("OpptyType_Original") == "UNMAPPED", "UNMAPPED")
         .when(F.col("OpptyType_Original") == "NEW FROM EXISTING", "NEW FROM EXISTING")
         .when(F.col("OpptyType_Original") == "NEW-NEW", "NEW-NEW")
         .when(F.col("OpptyType_Original") == "CROSS SELL", "CROSS SELL")
         .otherwise(F.col("OpptyType_Original"))) \
    .withColumn("OpptySubType",
         F.when(F.col("OpptyType_Original") == "GLOBAL FAC", F.col("OpptyType_Original"))
         .otherwise(F.col("OpptySubType"))) \
    .withColumn("ANZ_OpptyType",
         F.when((F.col("OpptyType") == "RENEWAL") & (F.col("Frequency").isin("RENEWAL", "RECURRING & ONE OFF", "UNMAPPED")), "RENEWAL")
         .when((F.col("OpptyType") == "RENEWAL") & (F.col("Frequency") == "ONE OFF"), "ONE OFF")
         .when((F.col("OpptyType") == "RENEWAL") & (F.col("Frequency").isNull()), "ONE OFF")
         .when(F.col("OpptyType").isin("NEW FROM EXISTING", "NEW-NEW", "GLOBAL FAC", "CROSS SELL", "WHOLESALE") & F.col("Frequency").isin("RECURRING", "RECURRING & ONE OFF"), "NEW BUSINESS")
         .when(F.col("OpptyType").isin("NEW FROM EXISTING", "NEW-NEW", "GLOBAL FAC", "CROSS SELL", "WHOLESALE") & (F.col("Frequency") == "ONE OFF"), "ONE OFF")
         .when(F.col("OpptyType").isin("NEW FROM EXISTING", "NEW-NEW", "GLOBAL FAC", "CROSS SELL", "WHOLESALE") & (F.col("Frequency") == "UNMAPPED"), "ONE OFF")
         .when(F.col("OpptyType").isin("NEW FROM EXISTING", "NEW-NEW", "GLOBAL FAC", "CROSS SELL", "WHOLESALE") & (F.col("Frequency").isNull()), "ONE OFF")
         .otherwise(F.col("OpptyType"))) \
    .drop("OpptySubType_Original", "OpptyType_Original")
         
# --- Final Finance Level Remapping (Tools 334, 338, 340, etc.) ---
asia_specialty_levels = ["GLOBAL FINANCIAL SOLUTIONS - GB", "TPV SINGAPORE TERRORISM", "P&C - GB", "AEROSPACE - GB", "TPV NA TERRORISM", "RISK & ANALYTICS - ASIA"]
asia_specialty_profit_centers = ["FINANCIAL SOLUTIONS - SINGAPORE", "SINGAPORE PROPERTY (P&C - GB)", "TERRORISM - SINGAPORE", "CAPTIVES - ASIA", "LARGE ACCOUNTS - SINGAPORE", "CRH - CLIMATE AND RESILENCE HUB TEAM", "NON GL - ASIA", "STRATEGIC RISK CONSULTING - ASIA", "FS SINGAPORE", "MARINE CARGO - ASIA SPECIALTY", "CLIMATE - ASIA", "GLOBAL FAC - ASIA SPECIALTY"]

final_enriched_df = final_enriched_df.withColumn("Finance_Level",
    F.when(F.col("Finance_Level").isin(asia_specialty_levels), "ASIA SPECIALTY")
     .when(F.col("Profit_Center").isin(asia_specialty_profit_centers) | (F.col("Profit_Center").like("%ASIA SPECIALTY%")), "ASIA SPECIALTY")
     .when(F.col("Profit_Center") == "JAPAN DESK - SINGAPORE", "ASIA DESK REPORTING")
     .when(F.col("Profit_Center") == "CONSTRUCTION - MALAYSIA", "MALAYSIA")
     .when((F.col("Finance_Level") == "ASIA DESK REPORTING") & (F.col("Data_Version") == "NON OPPORTUNITY SERVICE"), "ASIA - DESK REPORTING")
     .otherwise(F.col("Finance_Level"))
)

print("Batch 3 finished.")

# Check for NULL values
null_count = enriched_df.filter(F.col("Owner_Business_Name").isNull()).count()
print(f"NULL Owner_Business_Name count: {null_count}")

# Check the split with NULL handling
business_records_df = enriched_df.filter(
    F.col("Owner_Business_Name").isin(business_lines)
)

non_business_records_df = enriched_df.filter(
    (~F.col("Owner_Business_Name").isin(business_lines)) |
    F.col("Owner_Business_Name").isNull()
)

print(f"Fixed business count: {business_records_df.count()}")
print(f"Fixed non-business count: {non_business_records_df.count()}")
print(f"Fixed total: {business_records_df.count() + non_business_records_df.count()}")

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 9, Finished, Available, Finished, False)

Batch 3: Enriching data through mapping...
Batch 3 finished.
NULL Owner_Business_Name count: 0
Fixed business count: 244352
Fixed non-business count: 43532
Fixed total: 287884


In [8]:
print("Available columns:", final_enriched_df.columns)

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 10, Finished, Available, Finished, False)

Available columns: ['Account', 'CCY', 'CloseDate', 'Colleague_Involved', 'Colleague_Involved_Office', 'Colleague_Involved_UPN', 'Country__Account_Name__Account_', 'CreatedOn', 'Data_Version', 'Description', 'DunsNumber', 'Est__Revenue__LCY_', 'Est__Revenue__USD_', 'Finance_Level', 'First_Income_Date', 'FormName', 'GCID', 'GUID', 'Global_Parent_Account', 'Industry__Account_Name___Account_', 'Likelihood_of_Win', 'ModifiedOn', 'Opportunity_Owner', 'Opportunity_Owner_ProfitCenter', 'Opportunity_Owner_UPN', 'OpptyID', 'OpptyState', 'OpptyStatus', 'OpptySubType', 'OpptySummary', 'OpptyType', 'OriginatingLeadID', 'Parent_Account', 'Premium__USD_', 'Primary_SIC_Code__Account_Name__Account_', 'ProductClass', 'ProductSubClass', 'Profit_Center', 'Project_Start_Date', 'Service_Office', 'Service_Office_Country', 'Service_Region', 'Source_Campaign', 'TagList', 'Tiers', 'Wt__Revenue__LCY_', 'Wt__Revenue__USD_', 'Owner_Business_Name', 'Owner_Country', 'Owner_Lob_Name', 'Owner_Segment_Lob_Name', 'Map_t

In [9]:
# ======================================================================================
# Batch 4: Final Filtering and Selection
# --------------------------------------------------------------------------------------
# This replicates the final filtering and selection before output.
# ======================================================================================
print("Batch 4: Applying final filters and selecting columns...")

# Final selection and renaming is now done on the fully processed dataframe
final_df = final_enriched_df

# Select and rename final columns for the main output (FACT table)
final_columns = [
    "Account", "Parent_Account", "Global_Parent_Account", "OpptyID", "CloseDate",
    F.col("Country__Account_Name__Account_").alias("Account_Country"),
    "CreatedOn", "Data_Version", "FormName", "Description", "CCY",
    "Est__Revenue__LCY_",     # ✅ Matches (double underscores)
    "Est__Revenue__USD_",     # ✅ Matches (double underscores)
    "Wt__Revenue__LCY_",      # ✅ Matches (double underscores)
    "Wt__Revenue__USD_",      # ✅ Matches (double underscores)
    "Finance_Level", "First_Income_Date", "Frequency", "GCID", "GUID",
    "Industry",               # ✅ Already processed column (not the original long name)
    "Likelihood_of_Win", "ModifiedOn", "Opportunity_Owner", "OpptyState", "OpptyStatus",
    "OpptySummary", "OpptyType", "OpptySubType", "Pipeline_Phase",
    F.col("Primary_SIC_Code__Account_Name__Account_").alias("Primary_SIC_Code"),
    "ProductClass", "ProductSubClass", "Profit_Center", "Project_Start_Date", "Service_Office",
    "Service_Office_Country", "Service_Region", "Source_Campaign", "TagList", "Tiers",
    F.col("Colleague_Involved").alias("Owner"), "Owner_Business_Name", "Owner_Country", "Owner_Lob_Name",
    "Owner_Segment_Lob_Name", "Colleague_Involved_UPN", "Opportunity_Owner_UPN",
    "OriginatingLeadID", "n_GCID", "Opp_Link", "ANZ_OpptyType"
]

final_output_df = final_df.select(*final_columns)

# Optional: Remove extra mapping columns before final selection
final_df_cleaned = final_df.drop("Map_to_Business_Name", "Map_to_Segment_Name", "Mapping_to_County")
final_output_df = final_df_cleaned.select(*final_columns)
final_output_df = final_output_df.orderBy("Account")

print("Batch 4 finished.")
print(f"final_enriched_df count: {final_output_df.count()}")

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 11, Finished, Available, Finished, False)

Batch 4: Applying final filters and selecting columns...
Batch 4 finished.
final_enriched_df count: 287884


In [10]:
display(final_enriched_df[final_enriched_df['OpptyID'] == 1198436])

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c7ae78c-1f17-49de-8c00-c95658a57257)

In [11]:
# --- Moderate Fuzzy Matching with Blocking ---
print("Creating DIM_Account with blocking-based matching...")

# 1. Prepare data (same as above)
account_vertices = final_output_df.select(
    F.col("n_GCID").alias("id"),
    "Account",
    "GCID",
    "Industry",
    "Primary_SIC_Code",
    "Tiers"
).distinct()

account_vertices = advanced_preprocess(account_vertices, "Account")

# 2. Create blocking keys (first 4 characters)
account_vertices = account_vertices.withColumn(
    "block", F.substring(F.col("Account_processed"), 1, 4)
)

# 3. Within each block, pick the lexicographically first account as representative
window_spec = Window.partitionBy("block").orderBy("Account_processed", "Account")

representatives = account_vertices.withColumn(
    "rank", F.row_number().over(window_spec)
).filter(F.col("rank") == 1)

# 4. Create mapping from all accounts to their representatives
account_mapping = account_vertices.alias("all").join(
    representatives.alias("rep"),
    F.col("all.block") == F.col("rep.block")
).select(
    F.col("all.id").alias("original_id"),
    F.col("rep.id").alias("representative_id"),
    F.col("rep.Account").alias("representative_name")
)

# 5. Create final dimension table with all requested columns
dim_account_df = representatives.select(
    # Original columns
    F.col("Account"),                                           # Account
    F.col("GCID"),                                             # GCID
    F.col("Industry"),                                         # Industry
    F.col("Primary_SIC_Code").alias("Primary_SIC_Code"),      # Primary SIC Code
    F.col("Tiers"),                                           # Tiers

    # Additional requested columns
    F.col("id").alias("n.GCID"),                             # n.GCID (using the id which is n_GCID)
    F.upper(F.col("Account")).alias("UpperCaseAccountName"),  # UpperCaseAccountName
    F.col("Account").alias("GroupAccount")                    # GroupAccount (representative account)
)

print(f"DIM_Account created with {dim_account_df.count()} unique accounts")

# Show sample of the result
print("Sample of DIM_Account:")
dim_account_df.show(5, truncate=False)

print(f"DIM_Account created with {dim_account_df.count()} unique accounts")

StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 13, Finished, Available, Finished, False)

Creating DIM_Account with blocking-based matching...
DIM_Account created with 14083 unique accounts
Sample of DIM_Account:
+--------------------------------+-------+-----------------------------------+-------------------------------------------+------+-------+--------------------------------+--------------------------------+
|Account                         |GCID   |Industry                           |Primary_SIC_Code                           |Tiers |n.GCID |UpperCaseAccountName            |GroupAccount                    |
+--------------------------------+-------+-----------------------------------+-------------------------------------------+------+-------+--------------------------------+--------------------------------+
|11 11 GROUP PTY LTD             |3005289|LEISURE                            |4731 : FREIGHT TRANSPORTATION ARRANGEMENT  |TIER 4|3005289|11 11 GROUP PTY LTD             |11 11 GROUP PTY LTD             |
|122 MIDDLE INVESTMENT PTE. LTD. |2168448|FINANCIAL INSTITUTI

In [12]:
# ======================================================================================
# Batch 5: Generate Dimension Tables and Write Outputs
# --------------------------------------------------------------------------------------
# This batch creates and saves the final FACT and DIM tables.
# ======================================================================================
print("Batch 5: Generating and saving FACT and DIM tables...")

# --- FACT Table Output ---
# Equivalent to Alteryx Output Tool 186
fact_table_path = "APAC_Reporting_LH.APAC_Sales_Pipeline_FACT"
print(f"Writing main FACT table to Lakehouse path: {fact_table_path}")
final_output_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fact_table_path)
print("FACT table saved successfully.")


# --- DIM Table Outputs ---
dim_account_path = "APAC_Reporting_LH.DIM_Account"
print(f"Writing DIM_Account table to Lakehouse path: {dim_account_path}")
dim_account_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_account_path)
print("DIM_Account table saved successfully.")


# DIM_Other Tables
# This combines several small dimension outputs into one for simplicity.
dim_segment_df = final_df.select(
    F.col("Owner_Business_Name").alias("Business_Name"),
    F.col("Owner_Segment_Lob_Name").alias("Segment")
).distinct()

dim_segment_path = "APAC_Reporting_LH.DIM_Segment"
print(f"Writing DIM_Segment table to Lakehouse path: {dim_segment_path}")
dim_segment_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(dim_segment_path)
print("DIM_Segment table saved successfully.")

print("\n--- Workflow Translation Complete ---")


StatementMeta(, 3ba6ceba-a4eb-49bd-80fc-f705d4809e69, 14, Finished, Available, Finished, False)

Batch 5: Generating and saving FACT and DIM tables...
Writing main FACT table to Lakehouse path: APAC_Reporting_LH.APAC_Sales_Pipeline_FACT
FACT table saved successfully.
Writing DIM_Account table to Lakehouse path: APAC_Reporting_LH.DIM_Account
DIM_Account table saved successfully.
Writing DIM_Segment table to Lakehouse path: APAC_Reporting_LH.DIM_Segment
DIM_Segment table saved successfully.

--- Workflow Translation Complete ---
